In [5]:
from math import inf


def allocate_resource(income, resource):
    """
    income[k][u - 1] — доход k-го этапа при выделении u млн.
    resource — общий объем ресурса.

    Возвращает:
        optimum — максимальный суммарный доход;
        allocation — оптимальное распределение ресурса;
        value — таблица функций Беллмана;
        actions — таблица оптимальных управлений.
    """

    n = len(income)

    # Проверка исходных данных
    if not isinstance(resource, int) or resource < 0:
        raise ValueError("Ресурс должен быть неотрицательным целым числом.")

    if not income:
        raise ValueError("Необходимо задать хотя бы один этап.")

    if resource < n:
        raise ValueError("Недостаточно ресурса: каждому году нужен минимум 1 млн.")

    if resource > sum(len(row) for row in income):
        raise ValueError("Объем ресурса превышает доступный.")

    # Таблица функций Беллмана
    value = [
        [-inf] * (resource + 1)
        for _ in range(n + 1)
    ]

    # Таблица оптимальных управлений
    actions = [
        [[] for _ in range(resource + 1)]
        for _ in range(n)
    ]

    # После последнего этапа остаток ресурса должен быть равен нулю
    value[n][0] = 0

    # Обратный проход
    for k in range(n - 1, -1, -1):
        for s in range(resource + 1):

            # Минимум 1 млн, максимум — размер таблицы доходности
            max_u = min(s, len(income[k]))

            for u in range(1, max_u + 1):

                future = value[k + 1][s - u]

                if future == -inf:
                    continue

                candidate = income[k][u - 1] + future

                if candidate > value[k][s]:
                    value[k][s] = candidate
                    actions[k][s] = [u]

                elif candidate == value[k][s]:
                    actions[k][s].append(u)

    # Проверка существования допустимого решения
    if value[0][resource] == -inf:
        raise ValueError("Не существует допустимого распределения ресурса.")

    # Восстановление одного оптимального плана
    allocation = []
    s = resource

    for k in range(n):
        u = actions[k][s][0]
        allocation.append(u)
        s -= u

    # Проверка результата
    actual_income = sum(
        income[k][allocation[k] - 1]
        for k in range(n)
    )

    assert s == 0
    assert sum(allocation) == resource
    assert actual_income == value[0][resource]

    return value[0][resource], allocation, value, actions


In [6]:
income = [
    [8, 14, 19, 23, 26, 28, 30, 31, 32, 33],  # Год 1
    [7, 13, 18, 22, 25, 27, 29, 30, 31, 32],  # Год 2
    [20, 24, 27, 29, 31, 33, 34, 35, 36, 37]  # Год 3
]

resource = 12

optimum, allocation, value, actions = allocate_resource(
    income, resource
)

print("Максимальный суммарный доход:", optimum)
print("Оптимальное распределение:", allocation)

print("\nРаспределение по годам:")

for k in range(len(income)):
    allocated = allocation[k]
    result = income[k][allocated - 1]

    print(
        f"Год {k + 1}: "
        f"выделено {allocated} млн, "
        f"доход {result}"
    )

print("\nПроверка распределения:")
print("Всего выделено:", sum(allocation), "млн")
print("Суммарный доход:", sum(
    income[k][allocation[k] - 1]
    for k in range(len(income))
))

print("\nТаблицы функций Беллмана:")

for k in range(len(income)):
    print(f"V_{k + 1}:", value[k])
    print(f"A_{k + 1}:", actions[k])

Максимальный суммарный доход: 75
Оптимальное распределение: [4, 5, 3]

Распределение по годам:
Год 1: выделено 4 млн, доход 23
Год 2: выделено 5 млн, доход 25
Год 3: выделено 3 млн, доход 27

Проверка распределения:
Всего выделено: 12 млн
Суммарный доход: 75

Таблицы функций Беллмана:
V_1: [-inf, -inf, -inf, 35, 41, 47, 52, 57, 61, 65, 69, 72, 75]
A_1: [[], [], [], [1], [1, 2], [2], [2, 3], [3], [3, 4], [3, 4], [4], [4, 5], [4, 5]]
V_2: [-inf, -inf, 27, 33, 38, 42, 46, 49, 52, 54, 56, 58, 60]
A_2: [[], [], [1], [2], [3], [3, 4], [4], [4, 5], [5], [5, 6], [5, 6, 7], [5, 6, 7], [6, 7]]
V_3: [-inf, 20, 24, 27, 29, 31, 33, 34, 35, 36, 37, -inf, -inf]
A_3: [[], [1], [2], [3], [4], [5], [6], [7], [8], [9], [10], [], []]
